- https://chatgpt.com/s/t_6afef6643ef081919a901ed4b9c9c452

In [0]:
!pip install statsmodels

In [0]:
# Synthetic data generation
import numpy as np
import pandas as pd
import statsmodels.api as sm

np.random.seed(42)

n = 300

# Independent variables
X1 = np.random.uniform(1, 10, n)
X2 = np.random.uniform(2, 20, n)
X3 = np.random.uniform(5, 15, n)

# Actual regression equation
Y_true = 50 + 3*X1 + 5*X2 - 2*X3

# Heteroscedasticity:
# Error standard deviation increases with X1
sigma = 2 + 0.8*X1

# Autocorrelation: AR(1) errors
rho = 0.65
z = np.random.normal(0, 1, n)

u = np.zeros(n)
u[0] = z[0]

for i in range(1, n):
    u[i] = rho*u[i-1] + np.sqrt(1-rho**2)*z[i]

# Actual errors
errors = sigma*u

# Final dependent variable
Y = Y_true + errors

# Create dataframe
df = pd.DataFrame({
    "X1": X1,
    "X2": X2,
    "X3": X3,
    "Y": Y
})

print(df.head())


In [0]:
# OLS solution

import statsmodels.api as sm

X = df[["X1", "X2", "X3"]]
X = sm.add_constant(X)

y = df["Y"]

ols_model = sm.OLS(y, X).fit()

print(ols_model.summary())
print(ols_model.params)


In [0]:
# WLS Solution

# Error standard deviation
sigma = 2 + 0.8*df["X1"]

# Inverse variance weights
weights = 1 / sigma**2

# Fit WLS
wls_model = sm.WLS(
    y,
    X,
    weights=weights
).fit()

print(wls_model.summary())
print(wls_model.params)


In [0]:
# GLS Solution

# Standard deviation of errors
sigma = 2 + 0.8*df["X1"]

# AR(1) correlation
rho = 0.65

n = len(df)

indices = np.arange(n)

R = rho ** np.abs(
    indices[:, None] - indices[None, :]
)

# Covariance matrix
Omega = np.outer(sigma, sigma) * R

print(Omega[:5, :5])


In [0]:

gls_model = sm.GLS(
    y,
    X,
    sigma=Omega
).fit()

print(gls_model.summary())
print(gls_model.params)


In [0]:
# Evalution


from sklearn.metrics import mean_squared_error, mean_absolute_error

models = {
    "OLS": ols_model,
    "WLS": wls_model,
    "GLS": gls_model
}

results = []

for name, model in models.items():

    predictions = model.predict(X)
    residuals = y - predictions

    rmse = np.sqrt(
        mean_squared_error(y, predictions)
    )

    mae = mean_absolute_error(y, predictions)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": model.rsquared,
        "Residual_Correlation":
            np.corrcoef(
                residuals[:-1],
                residuals[1:]
            )[0, 1]
    })

results_df = pd.DataFrame(results)

print(results_df.round(3))


In [0]:
# Plots
from scipy.linalg import cholesky, solve_triangular
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

# GLS residuals
gls_resid = y - gls_model.predict(X)

# Cholesky decomposition: Omega = L @ L.T
L = cholesky(Omega, lower=True)

# Whiten residuals
white_resid = solve_triangular(
    L,
    gls_resid,
    lower=True
)

plot_acf(white_resid, lags=20)
plt.title("GLS Whitened Residual ACF")
plt.show()

In [0]:
# Diagnostic

from statsmodels.stats.diagnostic import (
    het_breuschpagan,
    acorr_ljungbox
)
from statsmodels.stats.stattools import (
    durbin_watson,
    jarque_bera
)

# OLS residuals
residuals = ols_model.resid

# 1. Heteroscedasticity test
bp = het_breuschpagan(
    residuals,
    X
)

print("Breusch-Pagan p-value:", bp[1])

# 2. Autocorrelation
print(
    "Durbin Watson:",
    durbin_watson(residuals)
)

print(
    acorr_ljungbox(
        residuals,
        lags=[5, 10],
        return_df=True
    )
)

# 3. Normality
jb = jarque_bera(residuals)

print("Jarque-Bera p-value:", jb[1])
